In [4]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.2)

In [5]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    four : int
    six: int

    strike_rate: float
    bpb : float
    boundary_percentage: float

In [6]:
def calculate_strike_rate(state: BatsmanState) -> BatsmanState:
    run = state["runs"]
    ball = state["balls"]

    state["strike_rate"] = (run/ball)*100
    return state

In [7]:
def calculate_bpb(state: BatsmanState) -> BatsmanState:
   four = state["four"]
   six = state["six"]
   ball = state["balls"]

   state["bpb"] = (four + six) / ball
   return state

In [8]:
def calculate_boundary_percentage(state: BatsmanState) -> BatsmanState:
    four = state["four"]
    six = state["six"]
    run = state["runs"]

    state["boundary_percentage"] = ((four*4) + (six*6)) / run * 100
    return state

In [9]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""The batsman scored {state['runs']} runs in {state['balls']} balls with a strike rate of {state['strike_rate']:.2f}.
He hit {state['four']} fours and {state['six']} sixes, resulting in a boundary percentage of {state['boundary_percentage']:.2f}% and a boundary per ball ratio of {state['bpb']:.2f}."""
    state["summary"] = summary
    return state

In [15]:
graph = StateGraph(BatsmanState)

graph.add_node("calculate_sr",calculate_strike_rate)
graph.add_node("calculate_bpb",calculate_bpb)
graph.add_node("calculate_boundary_percentage",calculate_boundary_percentage)
graph.add_node("summary",summary)

# we are going to define our edges 
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge("START", "calculate_boundary_percentage")


graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percentage", "summary")


graph.add_edge("summary", END)

# compile the graph

graph = StateGraph(BatsmanState)

graph.add_node("calculate_sr", calculate_strike_rate)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("summary", summary)

graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percentage")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percentage", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()


In [ ]:
initial_state = {'runs': 120, 'balls': 80, 'four': 10, 'six': 5, 'strike_rate': 0.0, 'bpb': 0.0, 'boundary_percentage': 0.0}

workflow.invoke(initial_state)